# Assignment 8: WordNet-based Word Sense Disambiguation (WSD)
## Improving Meaning Interpretation in Ambiguous Sentences

This notebook applies WordNet-based Word Sense Disambiguation to resolve the meaning
of ambiguous words depending on their context.

### Topics Covered:
1. Introduction to Word Sense Disambiguation
2. WordNet Synsets and Definitions
3. Lesk Algorithm (Simplified + NLTK)
4. WSD on Ambiguous Sentences
5. Comparison: with vs without WSD
6. Custom WSD Examples with Indian Context

## 1. Install and Import Libraries

In [1]:
!pip install nltk pywsd

     ---------------------------------------- 0.0/31.6 MB ? eta -:--:--
     ---------------------------------------- 0.0/31.6 MB ? eta -:--:--
      --------------------------------------- 0.5/31.6 MB 2.1 MB/s eta 0:00:15
      --------------------------------------- 0.8/31.6 MB 2.0 MB/s eta 0:00:16
     - -------------------------------------- 1.3/31.6 MB 2.1 MB/s eta 0:00:15
     -- ------------------------------------- 2.1/31.6 MB 2.3 MB/s eta 0:00:13
     --- ------------------------------------ 2.6/31.6 MB 2.4 MB/s eta 0:00:13
     ---- ----------------------------------- 3.4/31.6 MB 2.6 MB/s eta 0:00:11
     ----- ---------------------------------- 4.2/31.6 MB 2.8 MB/s eta 0:00:10
     ------ --------------------------------- 5.0/31.6 MB 2.9 MB/s eta 0:00:10
     ------- -------------------------------- 5.8/31.6 MB 3.0 MB/s eta 0:00:09
     -------- ------------------------------- 6.8/31.6 MB 3.2 MB/s eta 0:00:08
     --------- ------------------------------ 7.9/31.6 MB 3.4 MB/s


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import nltk
import warnings
warnings.filterwarnings('ignore')

nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('stopwords')

from nltk.corpus import wordnet as wn
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

print("Libraries imported successfully!")

Libraries imported successfully!


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ayush\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\ayush\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ayush\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ayush\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\ayush\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\ayush\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_percep

## 2. What is Word Sense Disambiguation?

Many words in English have **multiple meanings** (senses) depending on context.

Examples:
- **bank**: financial institution OR river bank
- **bat**:  flying mammal OR cricket/baseball bat
- **plant**: a tree/flower OR a factory
- **spring**: a season OR a coil spring OR to jump

WSD automatically selects the correct sense of a word based on the surrounding context.

In [3]:
# WordNet synsets for the word 'bank'
word = 'bank'
synsets = wn.synsets(word)

print(f"WordNet Synsets for '{word}':")
print("=" * 70)
for i, synset in enumerate(synsets, 1):
    print(f"\n  Sense {i}: {synset.name()}")
    print(f"    Definition : {synset.definition()}")
    print(f"    Examples   : {synset.examples()[:2]}")
    lemmas = [l.name() for l in synset.lemmas()]
    print(f"    Lemmas     : {lemmas}")

WordNet Synsets for 'bank':

  Sense 1: bank.n.01
    Definition : sloping land (especially the slope beside a body of water)
    Examples   : ['they pulled the canoe up on the bank', 'he sat on the bank of the river and watched the currents']
    Lemmas     : ['bank']

  Sense 2: depository_financial_institution.n.01
    Definition : a financial institution that accepts deposits and channels the money into lending activities
    Examples   : ['he cashed a check at the bank', 'that bank holds the mortgage on my home']
    Lemmas     : ['depository_financial_institution', 'bank', 'banking_concern', 'banking_company']

  Sense 3: bank.n.03
    Definition : a long ridge or pile
    Examples   : ['a huge bank of earth']
    Lemmas     : ['bank']

  Sense 4: bank.n.04
    Definition : an arrangement of similar objects in a row or in tiers
    Examples   : ['he operated a bank of switches']
    Lemmas     : ['bank']

  Sense 5: bank.n.05
    Definition : a supply or stock held in reserve for

## 3. Simplified Lesk Algorithm

The **Lesk Algorithm** picks the sense whose definition has the maximum overlap with the
context words (the other words in the sentence).

In [4]:
def simplified_lesk(word, sentence):
    """
    Simplified Lesk Algorithm for Word Sense Disambiguation.

    Args:
        word (str): The ambiguous word to disambiguate.
        sentence (str): The sentence providing context.

    Returns:
        best_synset: The synset with the highest context overlap.
    """
    stop_words = set(stopwords.words('english'))

    # Context: tokenize sentence, remove stop words and the target word
    context_tokens = [
        t.lower() for t in word_tokenize(sentence)
        if t.lower() not in stop_words and t.lower() != word.lower()
    ]
    context = set(context_tokens)

    best_synset = None
    max_overlap = -1

    for synset in wn.synsets(word):
        # Signature: words from definition + examples
        signature_text = synset.definition() + ' ' + ' '.join(synset.examples())
        signature = set(
            t.lower() for t in word_tokenize(signature_text)
            if t.lower() not in stop_words
        )

        overlap = len(context & signature)

        if overlap > max_overlap:
            max_overlap = overlap
            best_synset = synset

    return best_synset, max_overlap


print("Simplified Lesk Algorithm defined.")

Simplified Lesk Algorithm defined.


## 4. WSD on Classic Ambiguous Sentences

In [5]:
ambiguous_examples = [
    {
        'word': 'bank',
        'sentences': [
            "I went to the bank to deposit my salary.",
            "We sat by the river bank to watch the sunset."
        ]
    },
    {
        'word': 'bat',
        'sentences': [
            "The cricket player swung the bat and hit a six.",
            "A bat flew out of the cave at dusk."
        ]
    },
    {
        'word': 'plant',
        'sentences': [
            "The factory plant was shut down due to pollution.",
            "Water the plant every morning so it stays healthy."
        ]
    },
    {
        'word': 'spring',
        'sentences': [
            "The flowers bloom beautifully in spring.",
            "The spring in the mattress was broken."
        ]
    },
    {
        'word': 'rock',
        'sentences': [
            "The climber carefully gripped the rock and scaled the cliff.",
            "The band played rock music at the concert."
        ]
    }
]

print("WORD SENSE DISAMBIGUATION USING LESK ALGORITHM")
print("=" * 70)

for example in ambiguous_examples:
    word = example['word']
    print(f"\nWord: '{word}'")
    print("-" * 60)

    for sentence in example['sentences']:
        synset, overlap = simplified_lesk(word, sentence)
        print(f"  Sentence : {sentence}")
        if synset:
            print(f"  Sense    : {synset.name()}")
            print(f"  Meaning  : {synset.definition()}")
            print(f"  Overlap  : {overlap} common word(s)")
        else:
            print("  Sense    : Could not disambiguate.")
        print()

WORD SENSE DISAMBIGUATION USING LESK ALGORITHM

Word: 'bank'
------------------------------------------------------------
  Sentence : I went to the bank to deposit my salary.
  Sense    : bank.n.10
  Meaning  : a flight maneuver; aircraft tips laterally about its longitudinal axis (especially in turning)
  Overlap  : 1 common word(s)

  Sentence : We sat by the river bank to watch the sunset.
  Sense    : bank.n.01
  Meaning  : sloping land (especially the slope beside a body of water)
  Overlap  : 2 common word(s)


Word: 'bat'
------------------------------------------------------------
  Sentence : The cricket player swung the bat and hit a six.
  Sense    : bat.n.02
  Meaning  : (baseball) a turn trying to get a hit
  Overlap  : 1 common word(s)

  Sentence : A bat flew out of the cave at dusk.
  Sense    : bat.n.01
  Meaning  : nocturnal mouselike mammal with forelimbs modified to form membranous wings and anatomical adaptations for echolocation by which they navigate
  Overlap  

## 5. NLTK's Built-in Lesk Implementation

NLTK provides `lesk()` in `nltk.wsd` which also supports POS-constrained disambiguation.

In [6]:
from nltk.wsd import lesk

print("NLTK BUILT-IN LESK ALGORITHM")
print("=" * 70)

nltk_examples = [
    ("bank", "I deposited money at the bank near my house."),
    ("bank", "The fishermen sat on the bank of the river."),
    ("plant", "The nuclear plant provides electricity to the city."),
    ("plant", "She bought a cactus plant for her desk."),
    ("fly",   "The pilot can fly the aircraft through storms."),
    ("fly",   "A fly landed on the food left on the table."),
    ("light", "The room was filled with bright natural light."),
    ("light", "She preferred a light meal before the meeting.")
]

for word, sentence in nltk_examples:
    context = word_tokenize(sentence)
    synset  = lesk(context, word)
    print(f"  Word     : {word}")
    print(f"  Sentence : {sentence}")
    if synset:
        print(f"  Sense    : {synset.name()}")
        print(f"  Meaning  : {synset.definition()}")
    else:
        print("  Sense    : Not found.")
    print()

NLTK BUILT-IN LESK ALGORITHM
  Word     : bank
  Sentence : I deposited money at the bank near my house.
  Sense    : savings_bank.n.02
  Meaning  : a container (usually with a slot in the top) for keeping money at home

  Word     : bank
  Sentence : The fishermen sat on the bank of the river.
  Sense    : bank.n.01
  Meaning  : sloping land (especially the slope beside a body of water)

  Word     : plant
  Sentence : The nuclear plant provides electricity to the city.
  Sense    : plant.n.03
  Meaning  : an actor situated in the audience whose acting is rehearsed but seems spontaneous to the audience

  Word     : plant
  Sentence : She bought a cactus plant for her desk.
  Sense    : plant.n.01
  Meaning  : buildings for carrying on industrial labor

  Word     : fly
  Sentence : The pilot can fly the aircraft through storms.
  Sense    : fly.v.01
  Meaning  : travel through the air; be airborne

  Word     : fly
  Sentence : A fly landed on the food left on the table.
  Sense    :

## 6. POS-Constrained WSD

Providing the part-of-speech restricts the search to a specific category and improves accuracy.

In [7]:
def get_wordnet_pos(treebank_tag):
    """Map a Penn Treebank POS tag to a WordNet POS constant."""
    if treebank_tag.startswith('J'):
        return wn.ADJ
    elif treebank_tag.startswith('V'):
        return wn.VERB
    elif treebank_tag.startswith('N'):
        return wn.NOUN
    elif treebank_tag.startswith('R'):
        return wn.ADV
    return None


def wsd_with_pos(sentence, target_word):
    """
    Perform WSD with POS constraint.

    Returns the best synset for target_word in the given sentence.
    """
    tokens  = word_tokenize(sentence)
    pos_tags = pos_tag(tokens)

    wn_pos = None
    for token, tag in pos_tags:
        if token.lower() == target_word.lower():
            wn_pos = get_wordnet_pos(tag)
            break

    synset = lesk(tokens, target_word, pos=wn_pos)
    return synset, wn_pos


pos_examples = [
    ("interest", "The bank charges a high rate of interest on loans."),
    ("interest", "She has a deep interest in classical music."),
    ("well",     "He plays the guitar very well."),
    ("well",     "Workers dug a well to access groundwater."),
    ("fine",     "The weather is fine for a morning walk."),
    ("fine",     "He had to pay a fine for parking illegally.")
]

print("POS-CONSTRAINED WSD")
print("=" * 70)

for word, sentence in pos_examples:
    synset, wn_pos = wsd_with_pos(sentence, word)
    pos_label = {wn.NOUN: 'Noun', wn.VERB: 'Verb',
                 wn.ADJ: 'Adjective', wn.ADV: 'Adverb'}.get(wn_pos, 'Unknown')
    print(f"  Word     : '{word}' (detected POS: {pos_label})")
    print(f"  Sentence : {sentence}")
    if synset:
        print(f"  Sense    : {synset.name()}")
        print(f"  Meaning  : {synset.definition()}")
    else:
        print("  Sense    : Not found.")
    print()

POS-CONSTRAINED WSD
  Word     : 'interest' (detected POS: Noun)
  Sentence : The bank charges a high rate of interest on loans.
  Sense    : interest.n.01
  Meaning  : a sense of concern with and curiosity about someone or something

  Word     : 'interest' (detected POS: Noun)
  Sentence : She has a deep interest in classical music.
  Sense    : interest.n.01
  Meaning  : a sense of concern with and curiosity about someone or something

  Word     : 'well' (detected POS: Adverb)
  Sentence : He plays the guitar very well.
  Sense    : well.r.01
  Meaning  : (often used as a combining form) in a good or proper or satisfactory manner or to a high standard (`good' is a nonstandard dialectal variant for `well')

  Word     : 'well' (detected POS: Noun)
  Sentence : Workers dug a well to access groundwater.
  Sense    : well.n.01
  Meaning  : a deep hole or shaft dug or drilled to obtain water or oil or gas or brine

  Word     : 'fine' (detected POS: Adjective)
  Sentence : The weather i

## 7. WSD in Indian Context

Applying WSD to sentences with Indian-context vocabulary where word meanings are critical.

In [8]:
indian_context_examples = [
    {
        'word': 'fast',
        'sentences': [
            "During Navratri, many people observe a fast for nine days.",
            "The express train is very fast and reaches the destination quickly."
        ]
    },
    {
        'word': 'court',
        'sentences': [
            "The judge presided over the case in the High Court.",
            "The children played badminton on the court in the park."
        ]
    },
    {
        'word': 'cover',
        'sentences': [
            "The insurance policy will cover all medical expenses.",
            "The journalist was sent to cover the election results."
        ]
    },
    {
        'word': 'state',
        'sentences': [
            "Maharashtra is a prosperous state in western India.",
            "Please state your full name and address for the record."
        ]
    }
]

print("WSD IN INDIAN CONTEXT")
print("=" * 70)

for example in indian_context_examples:
    word = example['word']
    print(f"\nWord: '{word}'")
    print("-" * 60)
    for sentence in example['sentences']:
        synset, _ = wsd_with_pos(sentence, word)
        print(f"  Sentence : {sentence}")
        if synset:
            print(f"  Sense    : {synset.name()}")
            print(f"  Meaning  : {synset.definition()}")
        else:
            print("  Sense    : Not found.")
        print()

WSD IN INDIAN CONTEXT

Word: 'fast'
------------------------------------------------------------
  Sentence : During Navratri, many people observe a fast for nine days.
  Sense    : fast.n.01
  Meaning  : abstaining from food

  Sentence : The express train is very fast and reaches the destination quickly.
  Sense    : fast.r.01
  Meaning  : quickly or rapidly (often used as a combining form)


Word: 'court'
------------------------------------------------------------
  Sentence : The judge presided over the case in the High Court.
  Sense    : court.n.08
  Meaning  : a tribunal that is presided over by a magistrate or by one or more judges who administer justice according to the laws

  Sentence : The children played badminton on the court in the park.
  Sense    : court.n.02
  Meaning  : a room in which a lawcourt sits


Word: 'cover'
------------------------------------------------------------
  Sentence : The insurance policy will cover all medical expenses.
  Sense    : cover.v.02

## 8. Evaluation: WSD Accuracy

In [9]:
# Ground truth test set
test_cases = [
    {'word': 'bank',   'sentence': 'I deposited the cheque at the bank.',         'expected': 'bank.n.02'},
    {'word': 'bank',   'sentence': 'We relaxed on the bank of the river.',         'expected': 'bank.n.01'},
    {'word': 'bat',    'sentence': 'He scored a century with his bat.',            'expected': 'bat.n.01'},
    {'word': 'bat',    'sentence': 'The bat hung upside down in the cave.',        'expected': 'bat.n.02'},
    {'word': 'plant',  'sentence': 'The chemical plant caused air pollution.',     'expected': 'plant.n.02'},
    {'word': 'plant',  'sentence': 'She watered her plant every morning.',         'expected': 'plant.n.01'},
    {'word': 'spring', 'sentence': 'Birds return during spring season.',           'expected': 'spring.n.01'},
    {'word': 'spring', 'sentence': 'The spring in the clock snapped.',             'expected': 'spring.n.03'},
]

correct = 0

print("WSD ACCURACY EVALUATION")
print("=" * 70)
print(f"  {'#':<4} {'Word':<10} {'Predicted':<22} {'Expected':<22} {'Result'}")
print("  " + "-" * 65)

for i, tc in enumerate(test_cases, 1):
    tokens = word_tokenize(tc['sentence'])
    predicted_synset = lesk(tokens, tc['word'])
    predicted = predicted_synset.name() if predicted_synset else 'None'
    match = predicted == tc['expected']
    if match:
        correct += 1
    status = 'CORRECT' if match else 'WRONG'
    print(f"  {i:<4} {tc['word']:<10} {predicted:<22} {tc['expected']:<22} {status}")

accuracy = (correct / len(test_cases)) * 100
print(f"\n  Correct: {correct}/{len(test_cases)}")
print(f"  Accuracy: {accuracy:.1f}%")

WSD ACCURACY EVALUATION
  #    Word       Predicted              Expected               Result
  -----------------------------------------------------------------
  1    bank       savings_bank.n.02      bank.n.02              WRONG
  2    bank       bank.n.01              bank.n.01              CORRECT
  3    bat        bat.v.01               bat.n.01               WRONG
  4    bat        cricket_bat.n.01       bat.n.02               WRONG
  5    plant      plant.n.01             plant.n.02             WRONG
  6    plant      plant.n.01             plant.n.01             CORRECT
  7    spring     spring.n.01            spring.n.01            CORRECT
  8    spring     spring.n.01            spring.n.03            WRONG

  Correct: 3/8
  Accuracy: 37.5%


## 9. Summary and Conclusion

### What We Accomplished:

1. **Explored WordNet Synsets** - Examined multiple senses of ambiguous words.
2. **Implemented Simplified Lesk Algorithm** - Context overlap to pick the best sense.
3. **Used NLTK's `lesk()`** - Built-in word sense disambiguation.
4. **Applied POS-Constrained WSD** - Improved accuracy by limiting search to the correct POS.
5. **WSD in Indian Context** - Disambiguated words in culturally relevant sentences.
6. **Evaluated WSD Accuracy** - Measured performance against a ground truth test set.

### Key Concepts:
- **Synset** - A set of synonymous words representing one concept/sense in WordNet.
- **Lesk Algorithm** - Maximizes overlap between context words and synset definitions.
- **POS Constraint** - Reduces candidate synsets to those matching the grammatical role.

### Limitations:
- Lesk is a baseline algorithm; modern neural WSD models are significantly more accurate.
- Performance drops on rare or domain-specific senses not well covered in WordNet.
- Context window size affects quality; longer context generally improves results.